### Imports

In [9]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore
import faiss
import os

### RAG Pipeline

In [13]:
reader = SimpleDirectoryReader(input_dir = "./data_sample")
document = reader.load_data()

# Splitting sentences without losing sentence coherence
parser = SentenceSplitter(chunk_size = 256, chunk_overlap = 50)
nodes = parser.get_nodes_from_documents(document)

model = Ollama(model = "llama3.2:3b")
embedding = OllamaEmbedding(model_name="nomic-embed-text")

# Setting up non-volatile Faiss DB
faiss_dir = "./faiss_index"
os.makedirs(faiss_dir, exist_ok=True)
vector_store = FaissVectorStore(
    faiss_index=faiss.IndexFlatIP(768)
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(nodes, storage_context = storage_context, embed_model= embedding)
query_engine = index.as_query_engine(llm = model, embed_model = embedding, similarity_top_k = 5)

#### Question & Context

In [ ]:
questions = [
    "What is the optimal tilt angle for a site at 40 degrees north latitude?",
    "How often should solar panels be cleaned?",
    "What is the cycle life of LiFePO4 cells at 80% depth of discharge?",
    "What are the benefits of using LiFePO4 batteries in residential storage systems?",
    "What temperature range can lithium iron phosphate batteries operate in?",
    "What type of battery has the lowest initial cost but highest lifetime cost?",
    "What is the energy density of NMC batteries compared to other chemistries?",
    "How does shading affect solar panel output?",
    "Why are BMS components critical for lithium battery systems?",
    "What is the recommended roof load capacity for residential solar installations?"
]
ground_truths = [
    "The optimal tilt angle equals the latitude of the installation site. For a 40 degrees north latitude, the panel should be tilted at 40 degrees from horizontal.",
    "Solar panels should be cleaned every 2-3 months or after heavy dust storms or bird droppings to maintain efficiency and performance.",
    "LiFePO4 cells typically have a cycle life of 3,000-6,000 cycles at 80% depth of discharge.",
    "LiFePO4 batteries offer excellent thermal stability (non-flammable electrolyte), long cycle life, and high safety for residential storage applications.",
    "Lithium iron phosphate batteries can operate in temperatures from 0 to 45 degrees Celsius.",
    "Lead-acid (AGM/Gel) batteries have the lowest initial cost but higher lifetime costs due to frequent replacement requirements.",
    "NMC batteries have an energy density of 150-220 Wh/kg, which is higher than lead-acid (30-40 Wh/kg) but lower than LiFePO4 (90-160 Wh/kg).",
    "Even partial shading on a single cell can reduce the output of an entire string by up to 80%.",
    "The Battery Management System (BMS) is critical for lithium battery systems as it monitors and controls key parameters including cell voltage, temperature, current, and state of charge for safety and performance.",
    "Residential roofs must support at least 30 kg per square meter dead load plus 1.5 kN per square meter live load in snow regions."
]

### Result

In [16]:
for i in questions:
    print("\n" + "=" * 60)
    print("Q:", i)
    r = query_engine.query(i)
    print("A:", r.response)


Q: What is the optimal tilt angle for a site at 40 degrees north latitude?
A: The optimal tilt angle for a site at 40 degrees north latitude is 40 degrees from horizontal.

Q: How often should solar panels be cleaned?
A: According to the provided guidelines, solar panels should be cleaned quarterly. Additionally, it is recommended to use a soft brush and deionised water for cleaning, and never use abrasive cleaners or pressure washers above 40 bar, as these can crack the tempered glass cover.

Q: What is the cycle life of LiFePO4 cells at 80% depth of discharge?
A: Approximately 3,000 to 6,000 cycles.

Q: What are the benefits of using LiFePO4 batteries in residential storage systems?
A: LiFePO4 batteries offer several benefits in residential storage systems. They provide excellent thermal stability, making them non-flammable and reducing the risk of fires. Additionally, their high safety features, such as individual cell balancing and thermal management, ensure a reliable and secure 